# vWAN Routing Intent + NAT Gateway Bypass Test

**Testszenario:** Kann ein UDR mit Service Tag `WindowsVirtualDesktop` + Next-Hop `Internet` (NAT Gateway) den Azure Firewall umgehen, wenn vWAN Routing Intent aktiv ist?

**Setup:**
- vWAN Secured Hub mit Azure Firewall
- Routing Intent: PrivateTraffic + InternetTraffic → Firewall
- AVD Spoke VNet mit UDR: `WindowsVirtualDesktop` → Internet
- NAT Gateway auf dem AVD-Subnet

**Erwartung:**
- Default Traffic (0.0.0.0/0) → Firewall PIP
- WVD/AVD Service Tag Traffic → NAT Gateway PIP

## Variablen

In [ ]:
# Basis-Variablen
export PREFIX="cptdazavdvwan"
export RG="rg-${PREFIX}"
export VM="vm-avd-${PREFIX}"
export NIC="nic-avd-${PREFIX}"
echo "RG=$RG, VM=$VM, NIC=$NIC"

## 1. Deployment Outputs: Public IPs abrufen

**Erwartung:**
- `firewallPublicIp`: Die PIP des Azure Firewalls (für normalen Internet-Traffic)
- `natGatewayPublicIp`: Die PIP des NAT Gateways (für WVD-Traffic, falls Bypass funktioniert)

In [ ]:
# Deployment Outputs lesen
# Erwartung: Zwei unterschiedliche Public IPs
FW_PIP=$(az deployment group show -g $RG -n main --query properties.outputs.firewallPublicIp.value -o tsv)
NATGW_PIP=$(az deployment group show -g $RG -n main --query properties.outputs.natGatewayPublicIp.value -o tsv)

echo "Firewall Public IP:    $FW_PIP"
echo "NAT Gateway Public IP: $NATGW_PIP"
echo ""
echo "Interpretation:"
echo "  - Normaler Internet-Traffic sollte über $FW_PIP rausgehen (Routing Intent)"
echo "  - WVD/AVD Traffic sollte über $NATGW_PIP rausgehen (UDR + NAT GW)"

## 2. UDR (Route Table) Konfiguration prüfen

**Erwartung:** Zwei Routen in der Route Table:
1. `avd-wvd-direct-internet`: addressPrefix=`WindowsVirtualDesktop`, nextHopType=`Internet`
2. `test-udr-more-specific`: addressPrefix=`10.99.0.0/16`, nextHopType=`None`

In [ ]:
# Route Table Konfiguration anzeigen
# Erwartung: WindowsVirtualDesktop -> Internet UND 10.99.0.0/16 -> None
az network route-table show \
  --resource-group $RG \
  --name "rt-avd-${PREFIX}" \
  --query "routes[].{Name:name, Prefix:addressPrefix, NextHop:nextHopType}" \
  --output table

## 3. Effective Routes auf der AVD NIC

**Das ist der Schlüsseltest!**

**Erwartung (wenn Bypass funktioniert):**
- `0.0.0.0/0` → VirtualNetworkGateway/VirtualHub (Routing Intent → FW)
- WindowsVirtualDesktop IP-Ranges → Internet (UDR gewinnt für diese spezifischen IPs)
- `10.99.0.0/16` → None (UDR gewinnt über Routing Intent 10.0.0.0/8)

**Erwartung (wenn Bypass NICHT funktioniert):**
- `0.0.0.0/0` → VirtualNetworkGateway (Routing Intent überschreibt alles)
- Keine WindowsVirtualDesktop Routen sichtbar (UDR wird von Routing Intent überschrieben)
- `10.99.0.0/16` → None könnte trotzdem sichtbar sein (privater Bereich, nicht 0/0)

In [ ]:
# Effective Routes auf der NIC anzeigen
# DAS IST DER WICHTIGSTE TEST!
# Wenn WindowsVirtualDesktop IPs mit nextHopType "Internet" erscheinen,
# dann funktioniert der NAT-GW-Bypass trotz Routing Intent.
az network nic show-effective-route-table \
  --resource-group $RG \
  --name $NIC \
  --output table

In [ ]:
# Effective Routes gefiltert: Nur User-definierte Routen
# Erwartung: WindowsVirtualDesktop IPs und 10.99.0.0/16 als "User" source
az network nic show-effective-route-table \
  --resource-group $RG \
  --name $NIC \
  --query "value[?source=='User']" \
  --output table

In [ ]:
# Effective Routes gefiltert: Default Route (0/0)
# Erwartung: Source=VirtualNetworkGateway (von vWAN/Routing Intent injiziert)
az network nic show-effective-route-table \
  --resource-group $RG \
  --name $NIC \
  --query "value[?contains(addressPrefix[0],'0.0.0.0/0')]" \
  --output json

## 4. Live-Test: Outbound IP vom AVD VM prüfen

**Erwartung:**
- `curl ifconfig.me` → zeigt Firewall PIP (normaler Internet-Traffic geht über Routing Intent → FW)
- Falls FW den Traffic blockt: Timeout oder keine Antwort

Dieser Test zeigt den **Default-Pfad** (0/0 → FW). WVD-Traffic geht über andere Ports/Destinations.

In [ ]:
# Outbound IP Test via az vm run-command
# Erwartung: Die angezeigte IP sollte die FIREWALL PIP sein ($FW_PIP)
# Wenn NAT GW PIP ($NATGW_PIP) angezeigt wird, geht DEFAULT Traffic über NAT GW (unerwartet)
# Wenn Timeout: FW Application Rule blockt HTTP zu ifconfig.me
az vm run-command invoke \
  --resource-group $RG \
  --name $VM \
  --command-id RunPowerShellScript \
  --scripts "
    Write-Output '=== Outbound Public IP (Default Route) ==='
    try {
      \$ip = (Invoke-WebRequest -Uri http://ifconfig.me/ip -UseBasicParsing -TimeoutSec 15).Content
      Write-Output \"Public IP: \$ip\"
    } catch {
      Write-Output \"FEHLER/BLOCKED: \$_\"
      Write-Output 'Dies bedeutet: FW blockiert HTTP ohne explizite App Rule'
    }
  " \
  --query value[].message -o tsv

## 5. Live-Test: Verbindung zu WVD-Endpunkten prüfen

**Erwartung (wenn Bypass funktioniert):**
- TCP-Verbindung zu WVD-Endpunkten (Port 443) gelingt direkt über NAT GW
- Die Source-IP aus Sicht der WVD-Endpunkte ist die NAT GW PIP

**Erwartung (wenn Bypass NICHT funktioniert):**
- Verbindung geht über Firewall (FW muss FQDN-Rule für *.wvd.microsoft.com haben)
- Oder: Verbindung wird geblockt wenn FW keine passende Rule hat

In [ ]:
# Test: Konnektivität zu AVD/WVD Service Endpunkten
# Diese IPs/FQDNs gehören zum WindowsVirtualDesktop Service Tag
# Erwartung: Verbindung klappt (entweder direkt via NAT GW oder via FW)
az vm run-command invoke \
  --resource-group $RG \
  --name $VM \
  --command-id RunPowerShellScript \
  --scripts "
    Write-Output '=== Test: Verbindung zu WVD Endpunkten ==='
    
    \$endpoints = @(
      'rdweb.wvd.microsoft.com',
      'rdbroker.wvd.microsoft.com',
      'global.wvd.microsoft.com'
    )
    
    foreach (\$ep in \$endpoints) {
      Write-Output \"--- \$ep ---\"
      try {
        \$tcp = New-Object System.Net.Sockets.TcpClient
        \$tcp.ConnectAsync(\$ep, 443).Wait(5000) | Out-Null
        if (\$tcp.Connected) {
          Write-Output '  -> Verbindung OK (Port 443)'
          \$ip = [System.Net.Dns]::GetHostAddresses(\$ep) | Select-Object -First 1
          Write-Output \"  -> Resolved IP: \$ip\"
        } else {
          Write-Output '  -> TIMEOUT'
        }
        \$tcp.Close()
      } catch {
        Write-Output \"  -> FEHLER: \$_\"
      }
    }
  " \
  --query value[].message -o tsv

## 6. Traceroute-Vergleich: WVD-IP vs. normale Internet-IP

**Erwartung (wenn Bypass funktioniert):**
- Traceroute zu WVD-IP: Erster Hop ist NAT GW (kein FW dazwischen)
- Traceroute zu normaler IP: Erster Hop ist Firewall Private IP

**Hinweis:** Windows tracert kann unzuverlässig sein (ICMP wird oft geblockt). Aber der erste Hop zeigt den Unterschied.

In [ ]:
# Traceroute-Vergleich
# Erwartung: Unterschiedlicher erster Hop für WVD vs. normal
az vm run-command invoke \
  --resource-group $RG \
  --name $VM \
  --command-id RunPowerShellScript \
  --scripts "
    Write-Output '=== Resolve: WVD Endpoint IP ==='
    \$wvdIp = [System.Net.Dns]::GetHostAddresses('rdweb.wvd.microsoft.com') | Select-Object -First 1
    Write-Output \"WVD IP: \$wvdIp\"
    Write-Output ''
    Write-Output '=== Traceroute zu WVD IP (max 5 hops) ==='
    tracert -d -h 5 \$wvdIp
    Write-Output ''
    Write-Output '=== Traceroute zu 8.8.8.8 (max 5 hops) ==='
    tracert -d -h 5 8.8.8.8
  " \
  --query value[].message -o tsv

## 7. NAT Gateway Metrics: Wurde Traffic verarbeitet?

**Erwartung (wenn Bypass funktioniert):**
- `ByteCount` > 0: NAT Gateway hat tatsächlich Traffic verarbeitet
- `SNATConnectionCount` > 0: SNAT-Verbindungen wurden hergestellt

**Erwartung (wenn Bypass NICHT funktioniert):**
- Alle Metriken = 0: Kein Traffic geht über NAT GW

In [ ]:
# NAT Gateway Metrics der letzten 30 Minuten
# Erwartung: ByteCount > 0 wenn WVD Traffic über NAT GW geht
NATGW_ID=$(az network nat gateway show -g $RG -n "natgw-avd-${PREFIX}" --query id -o tsv)

echo "=== NAT Gateway: ByteCount (letzte 30 Min) ==="
az monitor metrics list \
  --resource "$NATGW_ID" \
  --metric "ByteCount" \
  --interval PT5M \
  --query "value[0].timeseries[0].data[-6:].{Time:timeStamp, Bytes:total}" \
  --output table 2>/dev/null || echo "Keine Daten (NAT GW wird nicht genutzt)"

echo ""
echo "=== NAT Gateway: SNATConnectionCount (letzte 30 Min) ==="
az monitor metrics list \
  --resource "$NATGW_ID" \
  --metric "SNATConnectionCount" \
  --interval PT5M \
  --query "value[0].timeseries[0].data[-6:].{Time:timeStamp, Connections:total}" \
  --output table 2>/dev/null || echo "Keine Daten"

## 8. Zusammenfassung & Interpretation

Führe diese Zelle nach allen Tests aus um die Ergebnisse zusammenzufassen:

In [ ]:
# Zusammenfassung
echo "============================================"
echo "  ROUTING INTENT + NAT GW BYPASS TEST"
echo "============================================"
echo ""

FW_PIP=$(az deployment group show -g $RG -n main --query properties.outputs.firewallPublicIp.value -o tsv 2>/dev/null)
NATGW_PIP=$(az deployment group show -g $RG -n main --query properties.outputs.natGatewayPublicIp.value -o tsv 2>/dev/null)

echo "Firewall PIP:    ${FW_PIP:-'nicht verfügbar'}"
echo "NAT Gateway PIP: ${NATGW_PIP:-'nicht verfügbar'}"
echo ""
echo "Prüfe effective routes auf User-definierte WVD-Routen..."
WVD_ROUTES=$(az network nic show-effective-route-table -g $RG -n $NIC \
  --query "value[?source=='User' && nextHopType=='Internet']" -o tsv 2>/dev/null)

if [ -n "$WVD_ROUTES" ]; then
  echo "✅ ERGEBNIS: UDR mit Internet next-hop ist in Effective Routes SICHTBAR"
  echo "   -> NAT Gateway Bypass funktioniert trotz Routing Intent!"
  echo "   -> WVD/RDP Traffic kann den Firewall umgehen."
else
  echo "❌ ERGEBNIS: Keine User-Route mit Internet next-hop in Effective Routes"
  echo "   -> Routing Intent überschreibt den UDR für Internet-Traffic."
  echo "   -> NAT Gateway Bypass funktioniert NICHT."
  echo "   -> Aller Traffic (inkl. WVD) geht über den Firewall."
fi
echo ""
echo "============================================"